# Lesson 1.3.2 - Clean Code, Modular Design & Documentation

## Learning Objectives
- Refactor monolithic scripts into testable modules.
- Use structured data types and clear naming conventions.
- Apply a practical documentation checklist for maintainability.


## Anti-Pattern Example
This intentionally mixes data cleaning, feature logic, and prediction in one function. It is hard to test and risky to modify.


In [1]:
from typing import List


def messy_pipeline(records: List[dict]) -> List[int]:
    out = []
    for r in records:
        age = 0 if r.get('age') is None else r.get('age')
        income = 0 if r.get('income') is None else r.get('income')
        # mixed concerns: cleaning + feature + decision
        score = age * 0.02 + income * 0.00001
        out.append(1 if score > 0.8 else 0)
    return out

sample = [{'age': 42, 'income': 120000}, {'age': None, 'income': 30000}]
print(messy_pipeline(sample))


[1, 0]


## Refactor Pattern: Separation of Concerns
Use small units with explicit contracts:
- `clean_record`
- `compute_score`
- `predict_label`

This improves testing and future extension.


In [2]:
from dataclasses import dataclass


@dataclass
class CustomerRecord:
    age: float
    income: float


def clean_record(raw: dict) -> CustomerRecord:
    return CustomerRecord(
        age=float(raw.get('age') or 0.0),
        income=float(raw.get('income') or 0.0),
    )


def compute_score(rec: CustomerRecord) -> float:
    return rec.age * 0.02 + rec.income * 0.00001


def predict_label(score: float, threshold: float = 0.8) -> int:
    return int(score > threshold)


def run_pipeline(raw_records: list[dict]) -> list[int]:
    labels = []
    for raw in raw_records:
        rec = clean_record(raw)
        score = compute_score(rec)
        labels.append(predict_label(score))
    return labels


print(run_pipeline(sample))


[1, 0]


## Reusable Unit Test Template
A minimal test set should cover:
- normal path
- missing values
- threshold boundary behavior


In [3]:
import unittest


class TestPipeline(unittest.TestCase):
    def test_clean_record_defaults(self):
        rec = clean_record({'age': None, 'income': None})
        self.assertEqual(rec.age, 0.0)
        self.assertEqual(rec.income, 0.0)

    def test_threshold_boundary(self):
        self.assertEqual(predict_label(0.8), 0)
        self.assertEqual(predict_label(0.81), 1)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestPipeline)
result = unittest.TextTestRunner(verbosity=1).run(suite)
print('tests_ok =', result.wasSuccessful())


.

.

----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


tests_ok = True


## Documentation Checklist Template
- [ ] Module purpose in one sentence.
- [ ] Input/output contracts documented.
- [ ] Assumptions and edge cases listed.
- [ ] Example usage included.
- [ ] Known limitations stated.

### Example Public Function Docstring
```python
def predict_label(score: float, threshold: float = 0.8) -> int:
    """Return binary decision from risk score.

    Args:
        score: Computed risk score.
        threshold: Decision threshold.

    Returns:
        1 if score > threshold else 0.
    """
```


## Business Case Studies & Exceptions
### Case 1: Duplicate Feature Logic
A team duplicated feature transformation logic in training and inference scripts. This caused train-serve skew. Refactoring into shared modules removed drift.

### Case 2: Missing Docs During Incident
On-call engineers could not triage quickly because data assumptions were undocumented. Adding runbooks + ownership metadata reduced incident resolution time.

### Exception
In short-lived hackathon prototypes, full documentation may be light. But once code is shared or reused, document contracts immediately.


## Interview Questions & Answers
1. **Q:** What does single responsibility mean in practice?  
   **A:** Each function/module should have one clear reason to change.
2. **Q:** Why refactor notebook logic into modules?  
   **A:** To make behavior testable, reusable, and safe for production.
3. **Q:** What is train-serve skew?  
   **A:** Difference between training and production data transformation/model behavior.
4. **Q:** What is a good module boundary in ML?  
   **A:** Ingestion, features, training, evaluation, serving.
5. **Q:** Why write tests for “simple” helper functions?  
   **A:** They encode assumptions that frequently break under data changes.
6. **Q:** What should README include for ML projects?  
   **A:** Setup, run steps, data contracts, metrics, limitations.
7. **Q:** When is OOP useful in ML code?  
   **A:** When managing complex stateful workflows or extensible components.
8. **Q:** How do you reduce hidden coupling?  
   **A:** Pass dependencies explicitly and avoid global mutable state.
9. **Q:** One clean-code anti-pattern in AI repos?  
   **A:** Large scripts mixing ETL, training, and serving code.
10. **Q:** What is the minimum doc quality bar?  
   **A:** Any new public module must define inputs, outputs, assumptions, and example usage.
